# Contrastive logistic regression

Reframes multi-label classification as one binary question — *does this text
match this label?* — over `(text_vec, label_vec)` pairs, then scores every
label at inference. Same data, seeded split, and evaluation as
`baseline_logreg.ipynb` / `setfit_training.ipynb`, so the report is comparable.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


def _combine(t, l):
    """Contrastive features for text/label vectors: concat + diff + prod."""
    return np.concatenate([t, l, t - l, t * l], axis=-1)


def embed_labels(label_dict, embed):
    """One row per label name with its embedding: label_key, vector_label."""
    keys = [l for labs in label_dict.values() for l in labs]
    assert len(keys) == len(set(keys)), 'label names must be unique across categories'
    vecs = np.asarray(embed.encode(keys, normalize_embeddings=True))
    return pd.DataFrame({'label_key': keys, 'vector_label': list(vecs)})


def make_contrastive_dataset(df, labels_df, *, n_neg=3, seed=42):
    """Cross texts × labels, keep all positives + n_neg sampled negatives per text.
    Long df: vector, vector_label, label (1 = text has this label, 0 = negative)."""
    keys = set(labels_df['label_key'])
    texts = df[['vector', 'labels']].reset_index(drop=True)
    texts = texts[texts['labels'].apply(lambda labs: any(k in keys for k in labs))].copy()
    texts['_tid'] = range(len(texts))
    pairs = texts.merge(labels_df, how='cross')
    pairs['label'] = [int(k in labs) for k, labs in zip(pairs['label_key'], pairs['labels'])]
    neg = pairs[pairs.label == 0].sample(frac=1, random_state=seed)   # shuffle negatives
    neg = neg[neg.groupby('_tid').cumcount() < n_neg]                 # first n_neg per text
    pairs = pd.concat([pairs[pairs.label == 1], neg]).sort_values('_tid')
    return pairs[['vector', 'vector_label', 'label']].reset_index(drop=True)


def to_multihot(labs, keys):
    s = set(labs)
    return np.array([int(k in s) for k in keys])


def predict_multihot(model, text_vectors, labels_df, threshold=0.5):
    """Score every (text, label) pair, threshold -> (n_texts, n_labels) multi-hot."""
    L = np.stack(labels_df['vector_label'])
    T = np.stack(list(text_vectors))
    n, m = len(T), len(L)
    X = _combine(np.repeat(T, m, axis=0), np.tile(L, (n, 1)))
    proba = model.predict_proba(X)[:, 1].reshape(n, m)
    return (proba >= threshold).astype(int)


def _smoke_test():
    rng = np.random.default_rng(0)
    labels_df = pd.DataFrame({'label_key': [f'c{i}' for i in range(5)],
                              'vector_label': [rng.normal(size=8) for _ in range(5)]})
    df = pd.DataFrame({'vector': [rng.normal(size=8) for _ in range(20)],
                       'labels': [list(rng.choice(labels_df.label_key, 2, replace=False))
                                  for _ in range(20)]})
    pairs = make_contrastive_dataset(df, labels_df, n_neg=2)
    X = _combine(np.stack(pairs['vector']), np.stack(pairs['vector_label']))
    y = pairs['label'].to_numpy()
    m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X, y)
    pred = predict_multihot(m, df['vector'], labels_df)
    assert pred.shape == (20, 5) and set(np.unique(pred)) <= {0, 1}
    print('smoke ok')


_smoke_test()

In [ ]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

label_dict = {
    'sentiment': ['positive', 'negative', 'neutral'],
    'topic': ['pricing', 'support', 'quality', 'shipping'],
}

embed = SentenceTransformer('all-MiniLM-L6-v2')
labels_df = embed_labels(label_dict, embed)
label_keys = labels_df['label_key'].tolist()

df = pd.read_csv('labeled_demo.csv')
df['labels'] = df['labels'].apply(lambda s: [x.split('::')[-1] for x in json.loads(s)])
df['vector'] = list(np.asarray(embed.encode(df['text'].tolist(), normalize_embeddings=True)))

# same split + seed as the other notebooks -> identical test set
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
print(len(train_df), 'train /', len(test_df), 'test')

In [ ]:
# contrastive dataset -> features -> ONE balanced logistic regression on the pairs
pairs = make_contrastive_dataset(train_df, labels_df)
X = _combine(np.stack(pairs['vector']), np.stack(pairs['vector_label']))
y = pairs['label'].to_numpy()
model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X, y)
print('trained on', len(pairs), 'pairs')

In [ ]:
pred = predict_multihot(model, test_df['vector'], labels_df)
truth = np.array([to_multihot(l, label_keys) for l in test_df['labels']])
print(classification_report(truth, pred, target_names=label_keys, zero_division=0))
print('exact-match row accuracy:', round(accuracy_score(truth, pred), 3))

In [ ]:
names = lambda row: sorted(k for k, v in zip(label_keys, row) if v)
results = pd.DataFrame({
    'text': test_df['text'].values,
    'labels': [sorted(l) for l in test_df['labels']],
    'predicted': [names(r) for r in pred],
})
results['is_correct'] = [set(t) == set(p)
                         for t, p in zip(results['labels'], results['predicted'])]
results